In [9]:
import torch

In [ ]:
import re
from collections import Counter, defaultdict

class BPETokenizer:
    def __init__(self, vocab_size: int = 10000, end_of_word: str = '</w>'):
        self.vocab_size : int   = vocab_size
        self.end_of_word: str   = end_of_word
        self.merge      : list  = list()
        self.vocab      : set   = set()
        self.char_to_id : dict  = dict()
        self.id_to_char : dict  = dict()
    
    def _get_word_parts(self, word: str) -> list[str]:
        return list(word) + [self.end_of_word]
    
    def _count_pairs(self, corpus: list[list[str]]) -> dict[tuple[str, str], int]:
        pair_counts = defaultdict(int)
        for word in corpus:
            for i in range(len(word)-1):
                pair_counts[word[i], word[i+1]] += 1
        return pair_counts

    def train(self, text: str):
        words = re.findall(r'\S+', text)
        corpus = [self._get_word_parts(w) for w in words]

        chars = set()
        for w in corpus:
            chars.update(w)
        self.vocab = chars
        self.vocab.add(self.end_of_word)

        for _ in range(self.vocab_size - len(self.vocab)):
            pair_counts = self._count_pairs(corpus)
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=dict.get)
            new_token = best_pair[0] + best_pair[1]
            self.merge.append((best_pair, new_token))
            self.vocab.add(new_token)

            new_corpus = []
            for word in corpus:
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word)-1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_corpus.append(new_word)
            corpus = new_corpus
        
        self.token_to_id = {tok: i for i, tok in enumerate(sorted(self.vocab))}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}
    
    def encode(self, text: str):
        words = re.findall(r'\S+', text)
        tokens = []
        for w in words:
            parts = self._get_word_parts(w)

            for (a, b), merge in self.merge:
                new_parts = []
                i = 0
                while i < len(parts):
                    if i < len(parts)-1 and parts[i] == a and parts[i+1] == b:
                        new_parts.append(merge)
                        i += 2
                    else:
                        new_parts.append(parts[i])
                        i += 1
                parts = new_parts
            tokens.extend(parts)
        return [self.token_to_id[t] for t in tokens if t in self.token_to_id]
    
    def decode(self, ids):
        return 